# Sentiment Analysis — Amazon Product Reviews

**Disciplina:** Engenharia de Software para IA e Frameworks Profundos  
**Projeto:** Classificação binária de sentimento em reviews de produtos Amazon  
**Dataset:** [Kaggle — yasserh/amazon-product-reviews-dataset](https://www.kaggle.com/datasets/yasserh/amazon-product-reviews-dataset)

Este notebook demonstra todas as entregas do projeto, organizadas por etapa.

---

In [ ]:
import sys
import os

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print("Raiz do projeto:", ROOT)

---
## Entrega 1 — Funções, Modularização e Tipagem
> **Etapas 1, 2 e 3 do professor**

O código foi organizado em módulos com responsabilidades separadas e todas as funções possuem type hints.

```
src/
├── data/loader.py          → carregamento e validação
├── preprocessing/transform.py → limpeza de texto e normalização de labels
├── models/model.py         → definição do modelo
├── training/train.py       → pipeline de treinamento
├── evaluation/metrics.py   → métricas de avaliação
├── inference/predict.py    → inferência em produção
└── utils/config.py         → constantes e hiperparâmetros
```

In [ ]:
import inspect
from src.data.loader import load_data, validate_columns, inspect_data
from src.preprocessing.transform import clean_text, normalize_label, preprocess_dataset

# Exibe as assinaturas tipadas das funções principais
funcoes = [
    load_data, validate_columns, inspect_data,
    clean_text, normalize_label, preprocess_dataset,
]
for fn in funcoes:
    sig = inspect.signature(fn)
    print(f"def {fn.__name__}{sig}")

In [ ]:
# Demonstração das funções de pré-processamento
exemplos = [
    ("This is GREAT!!!", 5),
    ("Terrible product...", 1),
    ("It's okay, I guess.", 3),
]
print(f"{'Texto original':<30} {'clean_text':<30} {'normalize_label'}")
print("-" * 75)
for texto, rating in exemplos:
    print(f"{texto:<30} {clean_text(texto):<30} {normalize_label(rating)}")

---
## Entrega 2 — NumPy
> **Etapa 4 do professor**

Uso de NumPy para:
- **Análise dimensional** dos dados com `np.min`, `np.max`, `np.mean`, `np.std`, `np.unique`
- **Vetorização** TF-IDF → matriz densa NumPy (`.toarray()`)
- **Normalização L2** por amostra com `np.linalg.norm`
- **Análise das dimensões** das matrizes resultantes

In [ ]:
from src.data.loader import load_data, inspect_data
from src.utils.config import DATA_PATH

df = load_data(DATA_PATH)
print("=== Análise NumPy do dataset ===")
inspect_data(df)

In [ ]:
import numpy as np
from src.preprocessing.transform import preprocess_dataset, vectorize_texts, normalize_features
from src.training.train import split_dataset
from src.utils.config import TEXT_COLUMN, LABEL_COLUMN, TEST_SIZE, MAX_FEATURES

df_clean = preprocess_dataset(df)
x_train, x_test, y_train, y_test = split_dataset(df_clean, TEXT_COLUMN, LABEL_COLUMN, TEST_SIZE)

# Vetorização TF-IDF → numpy dense
vectorizer, X_train_np = vectorize_texts(x_train, MAX_FEATURES)
X_test_np = vectorizer.transform(x_test).toarray()

print("=== Antes da normalização ===")
print(f"X_train shape : {X_train_np.shape}")
print(f"Norma média   : {np.linalg.norm(X_train_np, axis=1).mean():.4f}")
print(f"Valor max     : {np.max(X_train_np):.4f}")

# Normalização L2 com NumPy
X_train_np = normalize_features(X_train_np)
X_test_np  = normalize_features(X_test_np)

print("\n=== Após normalização L2 ===")
print(f"X_train shape : {X_train_np.shape}")
print(f"Norma média   : {np.linalg.norm(X_train_np, axis=1).mean():.4f}  ← deve ser ≈ 1.0")
print(f"Valor max     : {np.max(X_train_np):.4f}")

---
## Entrega 3 — PyTorch Parte 1: Tensores, Dataset e DataLoader
> **Etapa 5 do professor**

Conversão dos arrays NumPy para tensores PyTorch e criação dos DataLoaders.

In [ ]:
import torch
from src.training.train import to_tensors, create_dataloader
from src.utils.config import BATCH_SIZE

# Conversão NumPy → tensores PyTorch
X_train_t, y_train_t = to_tensors(X_train_np, y_train.to_numpy())
X_test_t,  y_test_t  = to_tensors(X_test_np,  y_test.to_numpy())

print("=== Tensores ===")
print(f"X_train : shape={X_train_t.shape}, dtype={X_train_t.dtype}")
print(f"y_train : shape={y_train_t.shape}, dtype={y_train_t.dtype}")
print(f"X_test  : shape={X_test_t.shape},  dtype={X_test_t.dtype}")
print(f"y_test  : shape={y_test_t.shape},  dtype={y_test_t.dtype}")

# TensorDataset + DataLoader
train_loader = create_dataloader(X_train_t, y_train_t, BATCH_SIZE)
val_loader   = create_dataloader(X_test_t,  y_test_t,  BATCH_SIZE, shuffle=False)

print("\n=== DataLoaders ===")
print(f"Batches treino    : {len(train_loader)}")
print(f"Batches validação : {len(val_loader)}")
print(f"Device disponível : {'cuda' if torch.cuda.is_available() else 'cpu'}")

# Verificação de um batch
X_batch, y_batch = next(iter(train_loader))
print(f"\nBatch exemplo: X={X_batch.shape}, y={y_batch.shape}")

---
## Entrega 4 — PyTorch Parte 2: Modelo, Treinamento e Inferência
> **Etapa 6 do professor**

Implementação completa do modelo neural:
- `SentimentMLP(nn.Module)` com `forward()`
- Função de perda: `BCEWithLogitsLoss`
- Otimizador: `Adam`
- Loop de treinamento e validação
- Salvamento e carregamento do modelo
- Script de inferência

In [ ]:
import torch.nn as nn
from src.models.model import build_model, save_model, load_model, predict
from src.utils.config import HIDDEN_DIM, OUTPUT_DIM, MODEL_PATH

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(MAX_FEATURES, HIDDEN_DIM, OUTPUT_DIM).to(device)

print("=== Arquitetura do SentimentMLP ===")
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal de parâmetros: {total_params:,}")

In [ ]:
from src.training.train import run_training
from src.utils.config import LEARNING_RATE, NUM_EPOCHS

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.BCEWithLogitsLoss()

print("=== Treinamento ===")
model = run_training(
    model, train_loader, val_loader,
    optimizer, criterion, NUM_EPOCHS, device
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from src.evaluation.metrics import evaluate_model, print_report

y_pred = predict(model, X_test_t, device)

print("=== Métricas de Avaliação ===")
metrics = evaluate_model(y_test.to_numpy(), y_pred)
print_report(metrics)

cm = confusion_matrix(y_test.to_numpy(), y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negative", "Positive"])
disp.plot(colorbar=False)
plt.title("Confusion Matrix — Modelo Final")
plt.tight_layout()
plt.show()

In [ ]:
# Salvamento e carregamento do modelo
save_model(model, MODEL_PATH)
print(f"Modelo salvo em: {MODEL_PATH}")

model_loaded = load_model(MODEL_PATH, MAX_FEATURES, HIDDEN_DIM, OUTPUT_DIM)
y_pred_loaded = predict(model_loaded, X_test_t, torch.device("cpu"))
metrics_loaded = evaluate_model(y_test.to_numpy(), y_pred_loaded)
print(f"\nAccuracy após reload: {metrics_loaded['accuracy']:.4f}  ← deve ser igual ao anterior")

In [ ]:
from src.inference.predict import predict_batch

exemplos = [
    "This product is absolutely amazing, I love it!",
    "Terrible quality, broke after one day. Total waste of money.",
    "Best purchase I have ever made, highly recommend!",
    "Very disappointed, does not work at all.",
]

print("=== Inferência em novos textos ===")
labels = predict_batch(exemplos, model, vectorizer, device)
for texto, label in zip(exemplos, labels):
    sentimento = "POSITIVE" if label == 1 else "NEGATIVE"
    print(f"  [{sentimento}] {texto}")

---
## Entrega 5 — Exercícios Intermediários: Comparação de Configurações
> **Etapa 7 do professor**

Comparação de pelo menos 2 configurações experimentais com análise técnica.

> Para a comparação completa (4 configurações + gráfico), abra `notebooks/experiments.ipynb`.

In [ ]:
import pandas as pd

configs = [
    {"name": "Baseline",   "hidden_dim": 256, "lr": 1e-3, "epochs": 10},
    {"name": "Rede maior", "hidden_dim": 512, "lr": 1e-3, "epochs": 10},
]

resultados = []
for cfg in configs:
    m = build_model(MAX_FEATURES, cfg["hidden_dim"], OUTPUT_DIM).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=cfg["lr"])
    crit = nn.BCEWithLogitsLoss()
    run_training(m, train_loader, val_loader, opt, crit, cfg["epochs"], device)
    preds = predict(m, X_test_t, device)
    met = evaluate_model(y_test.to_numpy(), preds)
    resultados.append({"Configuração": cfg["name"], **{k: round(v, 4) for k, v in met.items()}})

df_resultados = pd.DataFrame(resultados).set_index("Configuração")
print("=== Comparação de Configurações ===")
display(df_resultados)

df_resultados[["accuracy", "f1_score"]].plot(kind="bar", figsize=(8, 4), ylim=(0.85, 1.0))
plt.title("Comparação: Accuracy e F1-score")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("\n=== Análise técnica ===")
print("Rede maior (512 unidades) vs Baseline (256 unidades):")
diff = df_resultados.loc["Rede maior", "f1_score"] - df_resultados.loc["Baseline", "f1_score"]
print(f"  Diferença de F1: {diff:+.4f}")
print("  Dataset pequeno (~1000 amostras): mais parâmetros não garante ganho.")

---
## Entrega 6 — Testes Automatizados com unittest
> **Etapa 8 do professor**

Suite de testes cobrindo: carregamento de dados, pré-processamento, modelo e treinamento.

In [ ]:
import unittest

loader = unittest.TestLoader()
suite  = loader.discover(start_dir="tests", pattern="test_*.py")
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

print(f"\nTotal: {result.testsRun} testes | Falhas: {len(result.failures)} | Erros: {len(result.errors)}")

---
## Entrega 7 — Engenharia de Software: Visão, Requisitos e Arquitetura
> **Etapas 9, 10 e 11 do professor**

Documentação em `docs/`:

In [ ]:
from IPython.display import Markdown, display

for doc in ["docs/vision.md", "docs/requirements.md", "docs/architecture.md"]:
    print(f"\n{'='*60}")
    print(f"  {doc}")
    print(f"{'='*60}")
    with open(doc, encoding="utf-8") as f:
        display(Markdown(f.read()))

---
## Entrega 8 — Versionamento com Git
> **Etapa 12 do professor**

In [ ]:
import subprocess

print("=== Branch atual ===")
print(subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())

print("\n=== Histórico de commits ===")
log = subprocess.check_output(
    ["git", "log", "--oneline", "--graph", "--all", "-12"], text=True
)
print(log)

print("=== Contribuições ===")
shortlog = subprocess.check_output(
    ["git", "shortlog", "-sn", "--all"], text=True
)
print(shortlog)

---
## Entrega Final — Pipeline Completo
> **Etapa 13 do professor**

Execução completa do sistema via `main.py`.

In [ ]:
import subprocess
result = subprocess.run(
    ["python", "main.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

In [ ]:
import json

print("=== Artefatos gerados ===")
artefatos = [
    "data/processed/reviews_processed.csv",
    "results/models/model.pt",
    "results/models/vectorizer.pkl",
    "results/metrics/metrics.json",
    "results/figures/confusion_matrix.png",
]
for path in artefatos:
    existe = os.path.exists(path)
    tamanho = f"{os.path.getsize(path):,} bytes" if existe else "—"
    print(f"  {'✅' if existe else '❌'} {path:<45} {tamanho}")

print("\n=== Métricas finais ===")
with open("results/metrics/metrics.json") as f:
    for k, v in json.load(f).items():
        print(f"  {k:<12}: {v:.4f}")